# Results processing

#### 1. Wczytywanie

In [3]:
RESPONSES_DIR = "./data/responses"
TRANSCRIPTS_PATH = "./data/ted_talks_transcripts.tsv"
OUTPUT_DATASET_DIR = "./data/dataset"

TARGET_LANGUAGES = ["zh", "fi", "fr", "he", "ja", "pl"]

# klucz = nazwa katalogu w data/responses, dataset_col = nazwa kolumny w CSV
MODELS = {
    "QwQ-32B": {
        "dataset_col": "qwq",
        "full_name": "Qwen/QwQ-32B",
        "params": {"temperature": 1.0, "top_p": 1.0},
    },
}

# dla modeli spoza MODELS
DEFAULT_PARAMS = {"temperature": 1.0, "top_p": 1.0}

In [4]:
import json
import os
import re
import pandas as pd

pd.set_option("display.max_colwidth", None)

transcripts = pd.read_csv(TRANSCRIPTS_PATH, sep="\t")


def model_config(model_dir):
    """Metadane modelu z MODELS, z fallbackiem dla nieznanego katalogu."""
    cfg = MODELS.get(model_dir, {})
    return {
        "dataset_col": cfg.get("dataset_col", re.sub(r"\W+", "_", model_dir).strip("_").lower()),
        "full_name": cfg.get("full_name", model_dir),
        "params": cfg.get("params", DEFAULT_PARAMS),
    }


# każdy katalog w responses/ z plikiem responses.tsv to jeden model
model_responses = {}
for model_dir in sorted(os.listdir(RESPONSES_DIR)):
    path = f"{RESPONSES_DIR}/{model_dir}/responses.tsv"
    if not os.path.exists(path):
        continue
    df_m = pd.read_csv(path, sep="\t")
    df_m = df_m[df_m["response"].notna()]
    df_m = df_m[~df_m["response"].astype(str).str.startswith("EXCEPTION:")].copy()
    model_responses[model_dir] = df_m
    print(f"{model_dir}: {len(df_m)} odpowiedzi  |  per język: {df_m['target_lang'].value_counts().to_dict()}")

assert model_responses, f"Brak plików responses.tsv w {RESPONSES_DIR}"

QwQ-32B: 30 odpowiedzi  |  per język: {'zh': 5, 'fi': 5, 'fr': 5, 'he': 5, 'ja': 5, 'pl': 5}


In [5]:
PREVIEW_LANG = "pl"
PREVIEW_MODEL = next(iter(model_responses))

_preview = model_responses[PREVIEW_MODEL]
_preview = _preview[_preview["target_lang"] == PREVIEW_LANG]
if len(_preview):
    print(f"{PREVIEW_MODEL}  |  {PREVIEW_LANG}  |  {_preview.iloc[0]['id']}")
    print("=" * 80)
    print(_preview.iloc[0]["response"])
else:
    print(f"Brak odpowiedzi dla języka '{PREVIEW_LANG}' ({PREVIEW_MODEL})")

QwQ-32B  |  pl  |  pl--adam_grant_how_to_stop_languishing_and_start_finding_flow
Okay, I need to translate this entire TED Talk transcript from English to Polish, following all the given rules. Let me start by carefully reading through the instructions to make sure I don't make any mistakes. The user is a professional translator, so the translation needs to be accurate and match the style of the provided reference.

First, I should ensure that every single cue is translated, keeping the same structure: cue number, timestamp, and text. I must not skip any cues or abbreviate. Each text line must be translated, but I have to keep proper names and TED as is. Stage directions like (Laughter) need to be translated into Polish, so "(Biserne śmiechy)" or whatever the official term is for "Biserne śmiechy" in Polish. Wait, looking at the reference, they used "Applause" as "(Applause)" but in the example given, "(Laughter)" turned into "(Śmiech)" and "(Applause)" as "(Brawa)". Need to check cons

#### Narzedzia SRT

In [6]:
TS_RE = re.compile(r"\d{2}:\d{2}:\d{2}[,.]\d{1,3}\s*-->\s*\d{2}:\d{2}:\d{2}[,.]\d{1,3}")


def strip_think(text):
    """Usuwa sekcję rozumowania modelu, jeśli występuje; dla modeli bez <think> zwraca tekst bez zmian."""
    text = str(text)
    # domknięte bloki <think>...</think>
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # samo </think> bez otwarcia (ucięty reasoning)
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()


def extract_srt(text):
    text = str(text)
    text = re.sub(r"```[a-zA-Z]*\n?", "", text)

    lines = text.split("\n")
    ts_idxs = [i for i, l in enumerate(lines) if TS_RE.search(l)]
    if not ts_idxs:
        return ""

    first_ts = ts_idxs[0]
    start = first_ts - 1 if first_ts > 0 and lines[first_ts - 1].strip().isdigit() else first_ts
    return "\n".join(lines[start:]).strip()


def normalize_srt(text):
    blocks = []
    for raw in re.split(r"\n\s*\n", str(text).strip()):
        block_lines = [l for l in raw.split("\n") if l.strip()]
        ts_pos = next((i for i, l in enumerate(block_lines) if TS_RE.search(l)), None)
        if ts_pos is None:
            continue
        ts_line = block_lines[ts_pos].strip()
        body = [l.strip() for l in block_lines[ts_pos + 1:] if l.strip()]
        if not body:
            continue
        num = block_lines[ts_pos - 1].strip() if ts_pos > 0 and block_lines[ts_pos - 1].strip().isdigit() else str(len(blocks) + 1)
        blocks.append(f"{num}\n{ts_line}\n" + "\n".join(body))
    return "\n\n".join(blocks)


def clean_srt(text):
    return normalize_srt(extract_srt(text))

#### 2. Czyszczenie
`strip_think` ucina rozumowanie (jeśli model je dodaje), `clean_srt` wyciąga i normalizuje sam SRT. Ta sama ścieżka dla modeli myślących i zwykłych.

In [7]:
for model_dir, df_m in model_responses.items():
    df_m["cleaned"] = df_m["response"].apply(lambda t: clean_srt(strip_think(t)))
    empty = (df_m["cleaned"].str.len() == 0).sum()
    print(f"{model_dir}: wyczyszczono {len(df_m)} odpowiedzi  |  pustych po czyszczeniu: {empty}")

_sample = model_responses[PREVIEW_MODEL]
_sample = _sample[_sample["target_lang"] == PREVIEW_LANG]
if len(_sample):
    print("=" * 80)
    print(_sample.iloc[0]["cleaned"][:1500])

QwQ-32B: wyczyszczono 30 odpowiedzi  |  pustych po czyszczeniu: 1
1
00:00:00,413 --> 00:00:02,707
Wiem, że wszyscy macie listy rzeczy do zrobienia,

2
00:00:02,749 --> 00:00:07,503
ale tak nie znoszę marnowania czasu, że mam listy rzeczy *do nierobienia*.

3
00:00:07,545 --> 00:00:09,547
Nie przeglądaj mediów społecznościowych,

4
00:00:09,589 --> 00:00:11,257
nie sprawdzaj telefonu w łóżku

5
00:00:11,299 --> 00:00:12,800
i nie włączaj telewizora,

6
00:00:12,842 --> 00:00:15,678
chyba że już wiem, co chcę obejrzeć.

7
00:00:15,720 --> 00:00:19,057
Ale rok temu łamałem wszystkie te zasady.

8
00:00:19,098 --> 00:00:21,059
Zostawałem wciąż na nogach po północy,

9
00:00:21,059 --> 00:00:22,351
dzikim scrolowaniem po mediach,

10
00:00:22,351 --> 00:00:24,979
gry w *online Scrabble*,

11
00:00:24,979 --> 00:00:29,776
i oglądaniem cały sezonów seriali, których nawet nie uznawałem.

12
00:00:30,777 --> 00:00:32,653
Rano budziłem się w stanie oszołomienia

13
00:00:32,653 --> 00:00:35,573


#### 3. Podział na języki
Odpowiedzi każdego modelu rozbite po języku, zapis do osobnych TSV.

In [8]:
assert all("cleaned" in f.columns for f in model_responses.values()), "Brak kolumny 'cleaned' - uruchom sekcję 2."

lang_frames = {}  # (model, język) -> DataFrame
for model_dir, df_m in model_responses.items():
    split_dir = f"{RESPONSES_DIR}/{model_dir}"
    for lang in TARGET_LANGUAGES:
        df_lang = df_m[df_m["target_lang"] == lang].copy()
        if df_lang.empty:
            continue
        df_lang["slug"] = df_lang["id"].str.replace(f"^{lang}--", "", regex=True)
        lang_frames[(model_dir, lang)] = df_lang
        out_path = f"{split_dir}/{model_dir}__{lang}.tsv"
        df_lang.to_csv(out_path, sep="\t", index=False)
        print(f"  {model_dir} / {lang}: {len(df_lang)} wierszy -> {out_path}")

  QwQ-32B / zh: 5 wierszy -> ./data/responses/QwQ-32B/QwQ-32B__zh.tsv
  QwQ-32B / fi: 5 wierszy -> ./data/responses/QwQ-32B/QwQ-32B__fi.tsv
  QwQ-32B / fr: 5 wierszy -> ./data/responses/QwQ-32B/QwQ-32B__fr.tsv
  QwQ-32B / he: 5 wierszy -> ./data/responses/QwQ-32B/QwQ-32B__he.tsv
  QwQ-32B / ja: 5 wierszy -> ./data/responses/QwQ-32B/QwQ-32B__ja.tsv
  QwQ-32B / pl: 5 wierszy -> ./data/responses/QwQ-32B/QwQ-32B__pl.tsv


#### 4. Format do datasetu
Jeden CSV na język: stałe kolumny (wejście + referencja) i para kolumn surowa/cleaned dla każdego modelu, scalane po slugu. Obok schemat JSON z opisem kolumn. Dataset budowany od nowa z całego `data/responses` przy każdym uruchomieniu.

In [9]:
assert "lang_frames" in dir(), "Brak 'lang_frames' - uruchom sekcję 3."

AVAILABLE_LANGS = "en," + ",".join(TARGET_LANGUAGES)

FIXED_COLS = ["id", "available_lang_vers", "title", "article_revision_id",
              "summary", "en_source", "prompt"]

meta = transcripts.set_index("slug")


def build_schema(lang, model_cols):
    """Schemat JSON: stałe kolumny + kolumny modeli."""
    cols = [
        {"name": "id", "description": "Talk slug (ten sam dla danego talku we wszystkich jezykach)",
         "language_model": None, "cleaned": None},
        {"name": "available_lang_vers", "description": "Kody wszystkich dostepnych wersji jezykowych talku",
         "language_model": None, "cleaned": None},
        {"name": "title", "description": "Tytul TED talku", "language_model": None, "cleaned": None},
        {"name": "article_revision_id", "description": "Brak odpowiednika dla TED (puste)",
         "language_model": None, "cleaned": None},
        {"name": "summary", "description": f"Oficjalne tlumaczenie TED ({lang}) w formacie SRT (referencja)",
         "language_model": None, "cleaned": None},
        {"name": "en_source", "description": "Zrodlowa transkrypcja EN w formacie SRT (wejscie do tlumaczenia)",
         "language_model": None, "cleaned": None},
        {"name": "prompt", "description": "Prompt uzyty do wygenerowania odpowiedzi modelu",
         "language_model": None, "cleaned": None},
    ]
    for mc in model_cols:
        cols.append({
            "name": mc["name"],
            "description": f"{mc['model']} response" + (" (cleaned version)" if mc["cleaned"] else ""),
            "language_model": {"name": mc["full_name"], "parameters": mc["params"]},
            "cleaned": mc["cleaned"],
        })
    return {"data_file": {"lang": lang, "name": {"stem": f"{lang}_ted_data", "suffix": ".csv"}},
            "columns": cols}


langs_present = [l for l in TARGET_LANGUAGES if any(key[1] == l for key in lang_frames)]

for lang in langs_present:
    frames = {m: f for (m, l), f in lang_frames.items() if l == lang}

    # jeden wiersz na slug (suma ze wszystkich modeli)
    slugs = sorted({s for df_l in frames.values() for s in df_l["slug"]})
    base_rows = []
    for slug in slugs:
        m = meta.loc[slug] if slug in meta.index else None
        prompt = ""
        for df_l in frames.values():  # prompt ten sam dla modeli, bierzemy pierwszy
            hit = df_l.loc[df_l["slug"] == slug, "prompt"]
            if len(hit):
                prompt = hit.iloc[0]
                break
        base_rows.append({
            "id": slug,
            "available_lang_vers": AVAILABLE_LANGS,
            "title": (m["Ted talk title"] if m is not None else ""),
            "article_revision_id": "",
            "summary": (m[f"{lang}_timed"] if m is not None else ""),
            "en_source": (m["en_timed"] if m is not None else ""),
            "prompt": prompt,
        })
    out_df = pd.DataFrame(base_rows, columns=FIXED_COLS)

    # doklejamy kolumny modeli po slugu
    model_cols = []
    for model_dir, df_l in frames.items():
        cfg = model_config(model_dir)
        col = cfg["dataset_col"]
        part = (df_l.drop_duplicates("slug").set_index("slug")[["response", "cleaned"]]
                .rename(columns={"response": col, "cleaned": f"{col}__cleaned"}))
        out_df = out_df.merge(part, left_on="id", right_index=True, how="left")
        for name, cleaned in [(col, False), (f"{col}__cleaned", True)]:
            model_cols.append({"name": name, "model": model_dir, "full_name": cfg["full_name"],
                               "params": cfg["params"], "cleaned": cleaned})

    out_dir = f"{OUTPUT_DATASET_DIR}/{lang}"
    os.makedirs(out_dir, exist_ok=True)
    csv_path = f"{out_dir}/{lang}_ted_data.csv"
    schema_path = f"{out_dir}/{lang}_ted_schema.json"
    out_df.to_csv(csv_path, index=False)

    with open(schema_path, "w", encoding="utf-8") as fh:
        json.dump(build_schema(lang, model_cols), fh, ensure_ascii=False, indent=2)

    print(f"  {lang}: {len(out_df)} wierszy, modele: {sorted(frames)} -> {csv_path}  (+ schema)")

  zh: 5 wierszy, modele: ['QwQ-32B'] -> ./data/dataset/zh/zh_ted_data.csv  (+ schema)
  fi: 5 wierszy, modele: ['QwQ-32B'] -> ./data/dataset/fi/fi_ted_data.csv  (+ schema)
  fr: 5 wierszy, modele: ['QwQ-32B'] -> ./data/dataset/fr/fr_ted_data.csv  (+ schema)
  he: 5 wierszy, modele: ['QwQ-32B'] -> ./data/dataset/he/he_ted_data.csv  (+ schema)
  ja: 5 wierszy, modele: ['QwQ-32B'] -> ./data/dataset/ja/ja_ted_data.csv  (+ schema)
  pl: 5 wierszy, modele: ['QwQ-32B'] -> ./data/dataset/pl/pl_ted_data.csv  (+ schema)


In [10]:
ds = pd.read_csv(f"{OUTPUT_DATASET_DIR}/{lang}/{lang}_ted_data.csv")

In [17]:
print(ds.iloc[3]["qwq__cleaned"]) 

1
00:00:02,796 --> 00:00:04,556
(Gaworzenie dziecka)

2
00:00:20,076 --> 00:00:22,796
A gdybym wam powiedziała,

3
00:00:22,836 --> 00:00:27,756
że gra w kryjówkę może zmienić świat?

4
00:00:29,076 --> 00:00:31,356
Wydaje się niemożliwe, prawda?

5
00:00:31,876 --> 00:00:35,756
Jestem tu dziś, aby udowodnić, że to możliwe.

6
00:00:36,436 --> 00:00:38,116
Cześć, jestem Molly i mam siedem lat.

7
00:00:38,156 --> 00:00:40,436
A to mój mały przyjaciel, Ari.

8
00:00:40,636 --> 00:00:42,316
Powiedz „cześć”, Ari.

9
00:00:43,276 --> 00:00:44,596
Cześć.

10
00:00:45,116 --> 00:00:48,156
O, i to jest mój sąsiad, Amarjot.

11
00:00:48,756 --> 00:00:53,036
Musi teraz wynieść Ari, żeby się zaparowały na nasz eksperyment.

12
00:00:53,156 --> 00:00:55,516
Ale nie martwcie się, wrócą.

13
00:00:56,916 --> 00:01:00,516
Mój wykład hoje jest o silnych sprawach, których dorosli możecie zrobić.

14
00:01:00,556 --> 00:01:03,756
które w kształtu dzieci i dorosłych, na jakie się stajemy.

15
00:01:04,5

In [14]:
print(ds.iloc[2]["summary"][:7500]) 

1
00:00:00,876 --> 00:00:03,196
Ludzie nie widzą drzew.

2
00:00:03,236 --> 00:00:05,476
Mijają nas każdego dnia.

3
00:00:05,516 --> 00:00:09,516
Siedzą, śpią, palą, urządzają pikniki,

4
00:00:09,516 --> 00:00:11,916
i całują się ukradkiem w naszym cieniu.

5
00:00:11,916 --> 00:00:15,436
Zrywają nasze liście i rozkoszują się owocami.

6
00:00:15,476 --> 00:00:17,156
Łamią nasze gałęzie

7
00:00:17,196 --> 00:00:21,196
albo drążą ostrzami imiona ukochanych na naszych pniach,

8
00:00:21,236 --> 00:00:23,476
ślubując wieczną miłość.

9
00:00:24,116 --> 00:00:26,596
Wyplatają naszyjniki z naszych igieł

10
00:00:26,636 --> 00:00:29,396
i malują nasze kwiaty na obrazach.

11
00:00:29,396 --> 00:00:32,716
Rąbią nas na kłody, by ogrzać własne domy,

12
00:00:32,756 --> 00:00:35,076
a czasami nas ścinają,

13
00:00:35,116 --> 00:00:38,276
sądząc, że zasłaniamy widok.

14
00:00:38,276 --> 00:00:43,436
Robią z nas kołyski, korki do wina, gumę do żucia, rustykalne meble.

15
00:00:43,476 --> 